# Tabulate CWE data

In [51]:
import xml.etree.ElementTree as ET

xml_file = "cwec_v4.16.xml"

# Parse the XML file
tree = ET.parse(xml_file)
root = tree.getroot()

# Define the namespace mapping
namespace = {'cwe': "http://cwe.mitre.org/cwe-7"}

# Find all weaknesses
weaknesses = root.findall(".//cwe:Weaknesses/cwe:Weakness", namespace)

print(f"Found {len(weaknesses)} CWE entries.")


Found 965 CWE entries.


In [52]:
import pandas as pd

# List to store extracted data
data = []

# Iterate through the weaknesses for inspection
for weakness in weaknesses:  
    cwe_id = weakness.get("ID", "N/A")
    name = weakness.get("Name", "N/A")
    abstraction = weakness.get("Abstraction", "N/A")
    structure = weakness.get("Structure", "N/A")
    status = weakness.get("Status", "N/A")

    # Extract Description and Extended Description
    description = weakness.find("cwe:Description", namespace)
    extended_description = weakness.find("cwe:Extended_Description", namespace)

    desc_text = description.text.strip() if description is not None and description.text else "N/A"
    ext_desc_text = extended_description.text.strip() if extended_description is not None and extended_description.text else "N/A"

    # Extract Related Weaknesses
    related_weaknesses_section = weakness.find("cwe:Related_Weaknesses", namespace)
    
    related_weaknesses = []
    if related_weaknesses_section is not None:
        for rw in related_weaknesses_section.findall("cwe:Related_Weakness", namespace):
            related_cwe_id = rw.get("CWE_ID", "N/A")
            nature = rw.get("Nature", "N/A")
            view_id = rw.get("View_ID", "N/A")
            ordinal = rw.get("Ordinal", "N/A")

            # Append as a structured string
            related_weaknesses.append(f"CWE-{related_cwe_id} ({nature}, View: {view_id}, Ordinal: {ordinal})")

    # Convert list to a readable string format
    related_text = "; ".join(related_weaknesses) if related_weaknesses else "N/A"

    # Extract Weakness Ordinality
    ordinalities = [
        ordinality.text.strip() for ordinality in weakness.findall(".//cwe:Weakness_Ordinalities/cwe:Weakness_Ordinality/cwe:Ordinality", namespace)
        if ordinality is not None and ordinality.text
    ]
    ordinalities_text = ", ".join(ordinalities) if ordinalities else "N/A"

    # Extract Applicable Platforms - Language Name and Technology Class
    language_names = [
        p.get("Name", "N/A") for p in weakness.findall(".//cwe:Applicable_Platforms/cwe:Language", namespace) if p.get("Name")
    ]
    technology_classes = [
        p.get("Class", "N/A") for p in weakness.findall(".//cwe:Applicable_Platforms/cwe:Technology", namespace) if p.get("Class")
    ]

    # Remove duplicates and join values
    language_text = ", ".join(set(language_names)) if language_names else "N/A"
    technology_text = ", ".join(set(technology_classes)) if technology_classes else "N/A"

    # Extract Alternate Terms
    alternate_terms = [
        term.text.strip()
        for term in weakness.findall(".//cwe:Alternate_Terms/cwe:Alternate_Term/cwe:Term", namespace)
        if term is not None and term.text
    ]

    # Extract Alternate Terms with Descriptions (if available)
    term_descriptions = []
    for alt_term in weakness.findall(".//cwe:Alternate_Terms/cwe:Alternate_Term", namespace):
        term_element = alt_term.find("cwe:Term", namespace)
        desc_element = alt_term.find("cwe:Description", namespace)
        
        if term_element is not None and term_element.text:
            term_text = term_element.text.strip()
            if desc_element is not None and desc_element.text:
                term_text += f" ({desc_element.text.strip()})"
            term_descriptions.append(term_text)

    # Join multiple terms with ", "
    alternate_terms_text = ", ".join(alternate_terms) if alternate_terms else "N/A"
    alternate_terms_with_desc_text = ", ".join(term_descriptions) if term_descriptions else "N/A"

    # Extract Background Details
    background_details = [
        detail.text.strip() for detail in weakness.findall(".//cwe:Background_Details/cwe:Background_Detail", namespace)
        if detail is not None and detail.text
    ]
    background_details_text = " ".join(background_details) if background_details else "N/A"

    # Extract Modes of Introduction (Phase + Note)
    modes_of_intro = []
    for intro in weakness.findall(".//cwe:Modes_Of_Introduction/cwe:Introduction", namespace):
        phase = intro.find("cwe:Phase", namespace)
        note = intro.find("cwe:Note", namespace)

        phase_text = phase.text.strip() if phase is not None and phase.text else "N/A"
        note_text = note.text.strip() if note is not None and note.text else "N/A"

        modes_of_intro.append(f"{phase_text}: {note_text}")

    # Join multiple introductions with " | "
    modes_of_intro_text = " | ".join(modes_of_intro) if modes_of_intro else "N/A"

    # Extract Likelihood of Exploit
    likelihood_element = weakness.find("cwe:Likelihood_Of_Exploit", namespace)
    likelihood_text = likelihood_element.text.strip() if likelihood_element is not None and likelihood_element.text else "N/A"

    # Extract Common Consequences
    consequences_list = []
    for consequence in weakness.findall(".//cwe:Common_Consequences/cwe:Consequence", namespace):
        scopes = [scope.text.strip() for scope in consequence.findall("cwe:Scope", namespace) if scope.text]
        impacts = [impact.text.strip() for impact in consequence.findall("cwe:Impact", namespace) if impact.text]
        note = consequence.find("cwe:Note", namespace)

        scope_text = ", ".join(scopes) if scopes else "N/A"
        impact_text = ", ".join(impacts) if impacts else "N/A"
        note_text = note.text.strip() if note is not None and note.text else "N/A"

        consequences_list.append(f"Scope: {scope_text} | Impact: {impact_text} | Note: {note_text}")

    # Join multiple consequences with " || "
    consequences_text = " || ".join(consequences_list) if consequences_list else "N/A"

    # Extract Detection Methods
    detection_methods = []
    for method in weakness.findall(".//cwe:Detection_Methods/cwe:Detection_Method", namespace):
        method_name = method.find("cwe:Method", namespace)
        description = method.find("cwe:Description", namespace)
        effectiveness = method.find("cwe:Effectiveness", namespace)

        method_text = method_name.text.strip() if method_name is not None and method_name.text else "N/A"
        description_text = description.text.strip() if description is not None and description.text else "N/A"
        effectiveness_text = effectiveness.text.strip() if effectiveness is not None and effectiveness.text else "N/A"

        detection_methods.append(f"Method: {method_text} | Effectiveness: {effectiveness_text} | Description: {description_text}")

    # Join multiple methods with " || "
    detection_methods_text = " || ".join(detection_methods) if detection_methods else "N/A"


       # Extract Potential Mitigations
    mitigations_list = []
    for mitigation in weakness.findall(".//cwe:Potential_Mitigations/cwe:Mitigation", namespace):
        phase = mitigation.find("cwe:Phase", namespace)
        description_element = mitigation.find("cwe:Description", namespace)

        phase_text = phase.text.strip() if phase is not None and phase.text else "N/A"

        # Extract all paragraph elements inside Description without assuming 'xhtml:p'
        description_texts = [desc.text.strip() for desc in description_element if desc is not None and desc.text]
        description_text = " ".join(description_texts) if description_texts else "N/A"

        mitigations_list.append(f"Phase: {phase_text} | Description: {description_text}")

    # Join multiple mitigations with " || "
    mitigations_text = " || ".join(mitigations_list) if mitigations_list else "N/A"
    


    # Extract Demonstrative Examples
    examples_list = []
    
    for example in weakness.findall(".//cwe:Demonstrative_Examples/cwe:Demonstrative_Example", namespace):
        # Extract Intro_Text
        intro_text = example.find("cwe:Intro_Text", namespace)
        intro_text_str = intro_text.text.strip() if intro_text is not None and intro_text.text else "N/A"

        # Extract Body_Text (multiple possible)
        body_texts = [bt.text.strip() for bt in example.findall("cwe:Body_Text", namespace) if bt.text]
        body_text_str = " ".join(body_texts) if body_texts else "N/A"

        # Extract Example_Code from all child elements
        example_code_element = example.find("cwe:Example_Code", namespace)
        example_code_str = " ".join(
            [elem.text.strip() for elem in example_code_element if elem.text]
        ) if example_code_element is not None else "N/A"

        # Store extracted data as a formatted string
        examples_list.append(f"Intro: {intro_text_str} | Code: {example_code_str} | Body: {body_text_str}")

    # Join multiple Demonstrative Examples with " || " to keep them separate in output
    examples_text = " || ".join(examples_list) if examples_list else "N/A"
    


        # Extract Observed Examples
    observed_list = []
    
    for observed in weakness.findall(".//cwe:Observed_Examples/cwe:Observed_Example", namespace):
        reference = observed.find("cwe:Reference", namespace)
        description = observed.find("cwe:Description", namespace)
        link = observed.find("cwe:Link", namespace)

        reference_text = reference.text.strip() if reference is not None and reference.text else "N/A"
        description_text = description.text.strip() if description is not None and description.text else "N/A"
        link_text = link.text.strip() if link is not None and link.text else "N/A"

        observed_list.append(f"Reference: {reference_text} | Description: {description_text} | Link: {link_text}")

    # Join multiple observed examples with " || "
    observed_text = " || ".join(observed_list) if observed_list else "N/A"


        # Extract Related Attack Patterns (CAPEC IDs)
    attack_patterns = [
        rap.get("CAPEC_ID", "N/A")
        for rap in weakness.findall(".//cwe:Related_Attack_Patterns/cwe:Related_Attack_Pattern", namespace)
    ]

    # Join multiple CAPEC IDs with ", "
    attack_patterns_text = ", ".join(attack_patterns) if attack_patterns else "N/A"

     # Extract Mapping Notes
    mapping_notes = weakness.find(".//cwe:Mapping_Notes", namespace)
    
    if mapping_notes is not None:
        usage = mapping_notes.find("cwe:Usage", namespace)
        rationale = mapping_notes.find("cwe:Rationale", namespace)
        comments = mapping_notes.find("cwe:Comments", namespace)
        
        # Extract multiple reasons
        reasons = [
            reason.get("Type", "N/A")
            for reason in mapping_notes.findall(".//cwe:Reasons/cwe:Reason", namespace)
        ]
        reasons_text = ", ".join(reasons) if reasons else "N/A"

        usage_text = usage.text.strip() if usage is not None and usage.text else "N/A"
        rationale_text = rationale.text.strip() if rationale is not None and rationale.text else "N/A"
        comments_text = comments.text.strip() if comments is not None and comments.text else "N/A"

        mapping_notes_text = f"Usage: {usage_text} | Rationale: {rationale_text} | Comments: {comments_text} | Reasons: {reasons_text}"
    else:
        mapping_notes_text = "N/A"

    # Append to the list as a dictionary
    data.append({
        "CWE_ID": cwe_id,
        "Name": name,
        "Abstraction": abstraction,
        "Structure": structure,
        "Status": status,
        "Description": desc_text,
        "Extended_Description": ext_desc_text,
        "Related_Weaknesses": related_text,
        "Weakness_Ordinality": ordinalities_text,
        "Language_Name": language_text,
        "Technology_Class": technology_text,
        "Alternate_Terms": alternate_terms_text,
        "Alternate_Terms_With_Descriptions": alternate_terms_with_desc_text,
        "Background_Details": background_details_text,
        "Modes_Of_Introduction": modes_of_intro_text,
        "Likelihood_Of_Exploit": likelihood_text,
        "Common_Consequences": consequences_text,
        "Detection_Methods": detection_methods_text,
        "Potential_Mitigations": mitigations_text,
        "Demonstrative_Examples": examples_text,
        "Observed_Examples": observed_text,
        "Related_Attack_Patterns": attack_patterns_text,
        "Mapping_Notes": mapping_notes_text,
    })

# Convert to DataFrame
df = pd.DataFrame(data)

# Display the DataFrame
df


,CWE_ID,Name,Abstraction,Structure,Status,Description,Extended_Description,Related_Weaknesses,Weakness_Ordinality,Language_Name,...,Background_Details,Modes_Of_Introduction,Likelihood_Of_Exploit,Common_Consequences,Detection_Methods,Potential_Mitigations,Demonstrative_Examples,Observed_Examples,Related_Attack_Patterns,Mapping_Notes
0,1004,Sensitive Cookie Without 'HttpOnly' Flag,Variant,Simple,Incomplete,The product uses a cookie to store sensitive i...,The HttpOnly flag directs compatible browsers ...,"CWE-732 (ChildOf, View: 1000, Ordinal: Primary)",N/A,N/A,...,An HTTP cookie is a small piece of data attrib...,Implementation: N/A,Medium,Scope: Confidentiality | Impact: Read Applicat...,Method: Automated Static Analysis | Effectiven...,Phase: Implementation | Description: N/A,"Intro: In this example, a cookie is used to st...",Reference: CVE-2022-24045 | Description: Web a...,N/A,Usage: Allowed | Rationale: This CWE entry is ...
1,1007,Insufficient Visual Distinction of Homoglyphs ...,Base,Simple,Incomplete,The product displays information or identifier...,,"CWE-451 (ChildOf, View: 1000, Ordinal: Primary)",Resultant,N/A,...,N/A,Architecture and Design: This weakness may occ...,Medium,"Scope: Integrity, Confidentiality | Impact: Ot...",Method: Manual Dynamic Analysis | Effectivenes...,Phase: Implementation | Description: Use a bro...,"Intro: The following looks like a simple, trus...",Reference: CVE-2013-7236 | Description: web fo...,632,Usage: Allowed | Rationale: This CWE entry is ...
2,102,Struts: Duplicate Validation Forms,Variant,Simple,Incomplete,The product uses multiple validation forms wit...,"If two validation forms have the same name, th...","CWE-694 (ChildOf, View: 1000, Ordinal: Primary...",Primary,Java,...,N/A,Implementation: N/A,N/A,Scope: Integrity | Impact: Unexpected State | ...,N/A,Phase: Implementation | Description: N/A,Intro: These two Struts validation forms have ...,N/A,N/A,Usage: Allowed | Rationale: This CWE entry is ...
3,1021,Improper Restriction of Rendered UI Layers or ...,Base,Simple,Incomplete,The web application does not restrict or incor...,A web application is expected to place restric...,"CWE-441 (ChildOf, View: 1000, Ordinal: Primary...",N/A,N/A,...,N/A,Implementation: N/A,N/A,Scope: Access Control | Impact: Gain Privilege...,Method: Automated Static Analysis | Effectiven...,Phase: Implementation | Description: The use o...,N/A,Reference: CVE-2017-7440 | Description: E-mail...,"103, 181, 222, 504, 506, 587, 654",Usage: Allowed | Rationale: This CWE entry is ...
4,1022,Use of Web Link to Untrusted Target with windo...,Variant,Simple,Incomplete,The web application produces links to untruste...,When a user clicks a link to an external site ...,"CWE-266 (ChildOf, View: 1000, Ordinal: Primary)",N/A,JavaScript,...,N/A,Architecture and Design: This weakness is intr...,Medium,Scope: Confidentiality | Impact: Alter Executi...,Method: Automated Static Analysis | Effectiven...,Phase: Architecture and Design | Description: ...,"Intro: In this example, the application opens ...",Reference: CVE-2022-4927 | Description: Librar...,N/A,Usage: Allowed | Rationale: This CWE entry is ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
960,95,Improper Neutralization of Directives in Dynam...,Variant,Simple,Incomplete,The product receives input from an upstream co...,This may allow an attacker to execute arbitrar...,"CWE-94 (ChildOf, View: 1000, Ordinal: Primary)",Primary,"JavaScript, Python, Ruby, Perl, Java, PHP",...,N/A,Implementation: REALIZATION: This weakness is ...,Medium,Scope: Confidentiality | Impact: Read Files or...,Method: Automated Static Analysis | Effectiven...,Phase: Architecture and Design | Description: ...,Intro: edit-config.pl: This CGI script is used...,Reference: CVE-2024-4181 | Description: Framew...,35,Usage: Allowed | Rationale: This CWE entry is ...
961,96,Improper Neutralization of Directives in Stati...,Base,Simple,Draft,The product receives in

In [ ]:
df.to_excel("../../data_merge/Datasets/cwe_data.xlsx")